In [1]:
import pandas as pd

In [2]:
# Importing survey data and subsetting relevant columns

kr = pd.read_stata("Unzip Files/Childrens Recode/KHKR82FL.DTA", convert_categoricals=False)
kr = kr[["caseid", "bord", "v001", "v002", "v003", "v005","b19", "h50", "h61", "h62", "h63", "m13", "m14", "m17", "m70"]].copy()

# caseid
# bord - Birth Order
# v001 - cluster id
# v002 - household id
# v005 - sample weight
# b19 - current age of child (Months)
# h50 - Hep B vaccination at Birth
# h61 - Hep B 1
# h62 - Hep B 2
# h63 - Hep B 3
# m13 - timing of 1st antenatal check (months)
# m14 - number of antenatal visits during pregnancy
# m17 - delivery by caesarean section
# m70 - baby postnatal check

hr = pd.read_stata("Unzip Files/Household Recode/KHHR82FL.DTA", convert_categoricals=False)
hr = hr[["hhid", "hv001", "hv002", "hv009", "hv014", "hv015", "hv024", "hv025", "hv046"]].copy()

# hv001 - cluster id
# hv002 - household id
# hv009 - number of household members
# hv014 - number of children under 5
# hv015 - result of household interview
# hv024 - region
# hv025 - type of place of residence
# hv046 - translator used


ir = pd.read_stata("Unzip Files/Individual Recode/KHIR82FL.DTA", convert_categoricals=False)
ir = ir[["caseid", "v001", "v002", "v003", "v012", "v045c", "v106", "v120", "v121", "v130", "v155", "v157", "v158", "v159", "v171b", "v190", "v717"]].copy()

# v001 - cluster id
# v002 - household id
# v003 - respondants line number
# v012 - respondents age
# v045c - native language of respondant
# v106 - highest educational level
# v120 - household has: radio
# v121 - household has: television
# v130 - religion
# v155 - literacy
# v157 - frequency of reading newspaper or magazine
# V158 - frequency of listening to radio 
# V159 - frequency of watching television
# v171b - frequency of using internet last month
# v190 - wealth index combined
# v717 - respondent's occupation



In [4]:
# ------------------------------------------------------------------
# DHS-compliant merge: KR (children) + IR (mothers) + HR (household)
# ------------------------------------------------------------------

# 1. Merge children with their biological mothers (KR → IR)
merged_df = kr.merge(
    ir,
    on="caseid",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------
# 2. Merge in household-level characteristics (→ HR)
#    Use CHILD household IDs (v001_x, v002_x)
# ------------------------------------------------------------------

merged_df = merged_df.merge(
    hr,
    left_on=["v001_x", "v002_x"],
    right_on=["hv001", "hv002"],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------
# 3. Clean up duplicated identifiers
# ------------------------------------------------------------------

merged_df = (
    merged_df
    .rename(columns={
        "v001_x": "v001",
        "v002_x": "v002",
        "v003_x": "v003"
    })
    .drop(columns=[
        "v001_y", "v002_y", "v003_y",
        "hv001", "hv002"
    ])
)

# ------------------------------------------------------------------
# 4. Sanity checks (do not skip)
# ------------------------------------------------------------------

# Row count must remain unchanged
assert len(merged_df) == len(kr), "Row count changed after merge"

# Every child must be linked to a mother
assert merged_df["caseid"].notna().all(), "Some children missing mother linkage"

# Household variables should exist
assert merged_df["v190"].notna().any(), "Household variables not merged correctly"




In [5]:
print(merged_df.columns.tolist())


['caseid', 'bord', 'v001', 'v002', 'v003', 'v005', 'b19', 'h50', 'h61', 'h62', 'h63', 'm13', 'm14', 'm17', 'm70', 'v012', 'v045c', 'v106', 'v120', 'v121', 'v130', 'v155', 'v157', 'v158', 'v159', 'v171b', 'v173', 'v190', 'v717', 'hhid', 'hv009', 'hv014', 'hv015', 'hv024', 'hv025', 'hv046']


In [6]:
# Creating Vaccination Flags

def vaccination_flag(row):

    # Vaccination at Birth
    at_birth = True
    if row["h50"] == 0.0:
        at_birth = False

    # Vaccination Flag
    vac_flag = "Full"
    if row["b19"] < 2:
        # here child is eligable for no post birth jabs
        pass
        
    elif row["b19"] < 3:
        # here child is only elligable for first post birth jab
        if row["h61"] == 0.0:
            vac_flag = "None Post Birth"
            
    elif row["b19"] < 4:
        # here child is eligable for first two post birth jabs
        if row["h61"] == 0.0:
            vac_flag = "None Post Birth"
        elif row["h62"] == 0.0:
            vac_flag = "First"

    else:
        # here child is eligable for all post birth jabs    
        if row["h61"] == 0.0:
            vac_flag = "None Post Birth"
        elif row["h62"] == 0.0:
            vac_flag = "First"
        elif row["h63"] == 0.0:
            vac_flag = "Second"

    return at_birth, vac_flag

        

merged_df[["Vaccination at Birth", "Vaccination Flag"]] = merged_df.apply(vaccination_flag, axis=1, result_type='expand')


In [7]:
# Apply Mappings

def apply_all_mappings(df):

    df = df.copy()

    df = df.rename(columns={

    # Childrens Recode
        
    "caseid": "Mothers id",
    "bord": "Birth Order",
    "v001": "Cluster id",
    "v002": "Household id",
    "b19": "Childs Age",
    "h50": "Received Hep B at birth",
    "h61": "Hep B 1",
    "h62": "Hep B 2",
    "h63": "Hep B 3",
    "m13": "Timing of 1st antenatal check",
    "m14": "Number of antenatal visits during pregnancy",
    "m17": "Delivery by caesarean section",
    "m70": "Baby postnatal check",
    
    # Household Recode

    "hv009": "Number of household members",
    "hv014": "Number of children under 5",
    "hv015": "Result of household interview",
    "hv024": "Region",
    "hv025": "Type of place of residence",
    "hv046": "Translator used",

    # Inidividual Recode

    "v012": "Mother's age",
    "v045c": "Native language of respondent",
    "v106": "Highest educational level",
    "v120": "Household has: radio",
    "v121": "Household has: television",
    "v130": "Religion",
    "v155": "Literacy",
    "v157": "Frequency of reading newspaper or magazine",
    "v158": "Frequency of listening to radio",
    "v159": "Frequency of watching television",
    "v171b": "Frequency of using internet last month",
    "v190": "Wealth index combined",
    "v717": "Mother's occupation"
    })
    
    mapping_dicts = {

        # Childrens Recode
        
        "Received Hep B at birth": {
            0.0: "No",
            1.0: "Vaccination date on card",
            2.0: "Reported by mother",
            3.0: "Vaccination marked on card",
            8.0: "Don't know",
            9.0: "Missing"
        },
        "Hep B 1": {
            0.0: "No",
            1.0: "Vaccination date on card",
            2.0: "Reported by mother",
            3.0: "Vaccination marked on card",
            8.0: "Don't know",
            9.0: "Missing"
        },
        "Hep B 2": {
            0.0: "No",
            1.0: "Vaccination date on card",
            2.0: "Reported by mother",
            3.0: "Vaccination marked on card",
            8.0: "Don't know",
            9.0: "Missing"
        },
        "Hep B 3": {
            0.0: "No",
            1.0: "Vaccination date on card",
            2.0: "Reported by mother",
            3.0: "Vaccination marked on card",
            8.0: "Don't know",
            9.0: "Missing"
        },
        "Number of antenatal visits during pregnancy": {
            98.0: 0,
            99.0: 0,
            None: 0
        },
        "Timing of 1st antenatal check": {
            98.0: "Don't know",
            99.0: "Missing",
            None: 9.0
        },
        "Delivery by caesarean section": {
            0.0: "No",
            1.0: "Yes",
            9.0: "Missing"
        },
        
        # Household Recode

        "Result of household interview": {
            1:  "Completed",
            2:  "No Household member/no competent member at home",
            3:  "Entire Household absent for extended period of time",
            4:  "Postponed",
            5:  "Refused",
            6:  "Dwelling vacant or address not a dwelling",
            7:  "Dwelling destroyed",
            8:  "Dwelling not found",
            9:  "Other"
        },
        "Region": {
            1: "Banteay Meanchey",
            2: "Battambang",
            3: "Kampong Cham",
            4: "Kampong Chhnang",
            5: "Kampong Speu",
            6: "Kampong Thom",
            7: "Kampot",
            8: "Kandal",
            9: "Koh Kong",
            10: "Kratie",
            11: "Mondul Kiri",
            12: "Phnom Penh",
            13: "Preah Vihear",
            14: "Prey Veng",
            15: "Pursat",
            16: "Ratanak Kiri",
            17: "Siemreap",
            18: "Preah Sihanouk",
            19: "Stung Treng",
            20: "Svay Rieng",
            21: "Takeo",
            22: "Otdar Meanchey",
            23: "Kep",
            24: "Pailin",
            25: "Tboung Khmum"
        },
        "Type of place of residence": {
            1: "Urban",
            2: "Rural"
        },
        "Translator used": {
            0: "No", 
            1: "Yes"
        },

        # Individual Recode

        "Native language of respondent": {
            1: "English",
            2: "Khmer",
            3: "Chinese",
            4: "Thai",
            5: "Chan",
            6: "Vietnamese"
        },
        "Highest educational level": {
            0: "No education",
            1: "Primary",
            2: "Secondary",
            3: "Higher",
            9: None
        },
        "Household has: radio": {
            0: "No",
            1: "Yes",
            7: "Not a dejure resident",
            9: None
        },
        "Household has: television": {
            0: "No",
            1: "Yes",
            7: "Not a dejure resident",
            9: None
        },
        "Religion": {
            1: "Buddhist",
            2: "Moslem",
            3: "Christian",
            95: "No religion",
            96: "Other",
            99: None
        },
        "Literacy": {
            0: "Cannot read at all",
            1: "Able to read only parts of a sentence",
            2: "Able to read whole sentence",
            3: "No card with required language",
            4: "Blind/visually impaired",
            9: None
        },
        "Frequency of reading newspaper or magazine": {
            0: "Not at all",
            1: "Less than once a week",
            2: "At least once a week",
            3: "Almost every day",
            9: None
        },
        "Frequency of listening to radio": {
            0: "Not at all",
            1: "Less than once a week",
            2: "At least once a week",
            3: "Almost every day",
            9: None
        },
        "Frequency of watching television": {
            0: "Not at all",
            1: "Less than once a week",
            2: "At least once a week",
            3: "Almost every day",
            9: None
        },
        "Frequency of using internet last month": {
            0: "Not at all",
            1: "Less than once a week",
            2: "At least once a week",
            3: "Almost every day",
            9: None
        },
        "Wealth index combined": {
            1: "Poorest",
            2: "Poorer",
            3: "Middle",
            4: "Richer",
            5: "Richest"
        },
        "Mother's occupation": {
            0.0: "Not working",
            1.0: "Professional/technical/managerial",
            2.0: "Clerical",
            3.0: "Sales",
            4.0: "Agricultural - self employed",
            5.0: "Agricultural - employee",
            6.0: "Household and domestic",
            7.0: "Services",
            8.0: "Skilled manual",
            9.0: "Unskilled manual",
            98.0: "Don't know",
            99.0: None
        }
        
    }
    
    
    for col, mapping in mapping_dicts.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(df[col])
    
    return df

merged_df = apply_all_mappings(merged_df)

C:\Users\oscar\AppData\Local\Temp\ipykernel_37536\239382951.py:263: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].map(mapping).fillna(df[col])


In [8]:
desired_order = ["None Post Birth", "First", "Second", "Full"]
merged_df["Vaccination Flag"] = pd.Categorical(
    merged_df["Vaccination Flag"],
    categories=desired_order,
    ordered=True
)
merged_df = merged_df.sort_values(by="Vaccination Flag")

merged_df['wt'] = merged_df['v005'] / 1000000

In [9]:
merged_df.to_csv("cleaned_data_NA_included.csv", index=False)

In [ ]:
merged_df = merged_df.dropna(subset=["Vaccination_at_Birth", "Vaccination_Flag"])
merged_df.to_csv("cleaned_data.csv", index=False)